V 3.0

### 一、以下是单推理谓词的结构优先的整体流程示例

#### 1. 将 CSV 数据转换为 GraphLib 格式，这个格式中边是带标签的:

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import re
import sys
import math
import time
import tempfile
import traceback
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np
import json
from pythonProject.src.Structure_first.fastest_pipeline import FastestGraphConverter, FastestEstimateMerger
from pythonProject.src.Structure_first.graph_sample import FastestRunner
from pythonProject.src.Structure_first.precision_submatching import ExactSubgraphMatcher
from pythonProject.src.Structure_first.proxy_sample import ProxyStratifiedSampler, compute_T_true

# 一级测试数据集
datasets_name = "parler_data"
# 一级数据集下根据查询和图结构的差异划分的子测试数据集
# dataset_name = "dataset_three"
dataset_name = "dataset_one"
# 原始CSV数据路径
CSV_BASE_DIR = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/csv_data"
# 转换后GraphLib数据存放路径
Graph_Lib_Dir = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/data_graph"

# converter = FastestGraphConverter(CSV_BASE_DIR,Graph_Lib_Dir)
# converter.run_without_author_user_post()
# converter.remove_edge_labels()

#### 2.1同时调用准确的子图匹配算法得到真实结果的基数，这些准确子图匹配算法只能处理边不带标签的图

##### 2.1.1 首先基于原来边带标签的图，精简成边不带标签的图

In [ ]:
# converter = FastestGraphConverter(CSV_BASE_DIR,Graph_Lib_Dir)
# converter.simplify_graph_merge_edges_update_degree(input_path=Graph_Lib_Dir+'/parler.graph',
#                                                    output_path='/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/data_graph/parler.graph')

##### 2.1.2 然后对精简后的数据图调用准确子图匹配算法进行匹配，这里得到的GT对应的查询图，应该与Fastest对应的查询图等价（就算边没有标签也没有影响，且两点不存在多条边）

                                        六千万基数： 准确匹配需要2min左右
                                        1.2亿基数： 准确匹配需要4min左右
                                        2.4亿基数： 准确匹配需要8min左右
                                        10亿基数：  准确匹配需要40min左右

In [ ]:
matcher = ExactSubgraphMatcher(
    exe_path="/home/wangshuo/projects/SubgraphMatching/build/matching/SubgraphMatching.out",
    default_args=["-filter", "GQL", "-order", "GQL", "-engine", "LFTJ", "-num", "MAX"],
    timeout=3000,
)

matcher.run_batch(
    data_graph=f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph/parler.graph",
    query_graph_dir=f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/query_graph",
    output_dir=f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/ground_truth",
)

#### 2.2 基于GraphLib数据，调用Fastest进行树采样或图采样，得到特定标签下所有或部分点的估计值:
（1) 对于随机生成的查询可以先调用Fastest估计基数，选择基数合适的查询作为实验对象 --- 对于parler_ans.txt中的数据可以先随机生成
（2）正常流畅下，需要先调用精确的子图匹配算法，得到真实基数结果存放到parler_ans.txt，然后再调用Fastest进行估计

In [3]:
# 初始化 Runner
runner = FastestRunner(build_dir="/home/wangshuo/projects/FaSTest-main/build")

# dataset_name = "dataset_one"
current_budget = 20000
infer_label = 1  # 对应 Post 节点的标签

# 调用 run 方法
code, output = runner.run(
    dataset=dataset_name,
    root_label=infer_label,           # 必须指定推理节点的标签
    sample_budget=current_budget,     # 设置预算
    # estimate_with_predicate=True      # <--- 开启单推理谓词模式
    estimate_with_predicate=False      # <--- 关闭单推理谓词模式
)

🚀 正在运行: /home/wangshuo/projects/FaSTest-main/build/Fastest -d dataset_one --ROOT_LABEL 1 --SAMPLE_BUDGET 20000

[CLI] Root label set to 1
[CLI] Sample budget set to 20000
[info] SAMPLE_BUDGET:20000
[Info] Cleared old estimate file: /home/wangshuo/resource/datasets/parler_data/dataset_one/results/in_estimateW_result.txt
[info] args:0x7ffcf25dca58
/home/wangshuo/resource/datasets/parler_data/dataset_one/ground_truth/parler_ans.txt
[Info] Reading core labels configuration from: /home/wangshuo/resource/datasets/parler_data/dataset_one/data_graph/user_custom_labels.txt
[Info] Core labels configuration file not found. Running in global estimation mode only.
[infor] root label1
Constructing Candidate Space: 12 20
[info] Query Size:16

Start Processing (Global) /home/wangshuo/resource/datasets/parler_data/dataset_one/query_graph/query_dense_1_1.graph
[Info] Using user-specified root label 1 -> query node 1
[Check] Sum(node_est) = 129165, est_total = 129165, ratio = 1
[INFO] sv_out_path /home/w

#### 2.3 生成推理节点文件infer_node.txt：
(1)得到准确的子图匹配结果后，要计算最终的真实的答案ST_IF_Truth，下直称T_truth,需要知道每个查询图中，代表推理谓词节点的顶点编号
(2)一个查询图的格式如下，三个顶点按照id依次编号为u0,u1,u2,接下来我想读取/home/wangshuo/resource/datasets/parler_data/dataset_one/query_graph文件夹下的所有查询文件，将标签为1的顶点所对应的标号保存到infer_node.txt，以下面查询为例要保存u1到文件中，
                                            t 3 3
                                            v 0 0 2
                                            v 1 1 2
                                            v 2 2 2
                                            e 1 2
                                            e 2 0
                                            e 1 0

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
extract_infer_node.py
从 query_graph 文件夹下的所有查询 .graph 文件中，
提取 label==1 的顶点编号，写入 infer_node.txt。
"""

import os

def extract_infer_node(dataset_name: str) -> str:
    """
    从指定数据集的 query_graph 文件夹下提取所有查询图中 label==1 的节点，
    并保存到 data_graph/infer_node.txt 文件中。

    参数:
        dataset_name (str): 数据集名称，如 "parler_data" 或 "dataset_two"

    返回:
        str: 输出文件路径
    """

    # === 配置路径 ===
    query_dir = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/query_graph"
    out_path = os.path.join(
        f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph",
        "infer_node.txt"
    )

    # === 收集所有 .graph 文件 ===
    graph_files = []
    for root, _, files in os.walk(query_dir):
        for f in files:
            if f.endswith(".graph"):
                graph_files.append(os.path.join(root, f))
    graph_files.sort()

    print(f"[INFO] Found {len(graph_files)} query graph files under {query_dir}")

    # === 解析每个文件，提取 label==1 的节点编号 ===
    results = []
    for path in graph_files:
        with open(path, "r") as f:
            for line in f:
                line = line.strip()
                if not line or not line.startswith("v "):
                    continue
                parts = line.split()
                if len(parts) >= 3:
                    vid = parts[1]
                    label = parts[2]
                    if label == "1":
                        results.append(f"u{vid}")
                        break  # 每个查询只有一个 label==1，提取后退出

    # === 写入输出文件 ===
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as fout:
        for r in results:
            fout.write(r + "\n")

    print(f"[DONE] Saved {len(results)} infer nodes to {out_path}")
    return out_path


# === 如果直接运行此脚本 ===
if __name__ == "__main__":
    dataset_name = "dataset_one"  # ✅ 修改为你的数据集名
    extract_infer_node(dataset_name)


### 3 - 4:
"""
处理 multi-query 的 Fastest 输出(in_estimateW_result.txt)：
  - 解析多个 Query 的每节点估计值（支持多个 Query 块）
  - 根据 INFER_NODE_FILE 中每行指定的 gt_match_col（例如 u1,u2）按顺序对应每个 Query
  - 将每个 Query 的 estimate 列并入 post_with_estimate.csv（列名 estimate__<query_basename>）
  - 针对每个 Query 用 ProxyStratifiedSampler 做评估（使用 compute_T_true 得到 T_true）
  - 输出 summary CSV / TXT

"""

#### 3.处理 Fastest 输出结果，将估计值与目标节点的文件合并

In [4]:

from pythonProject.src.Structure_first.proxy_sample import ProxyStratifiedSampler, compute_T_true

# ===========================
# = 配置区域（按需修改） =
# ===========================
# dataset_name = 'dataset_three'
print(dataset_name)

# Fastest对所有查询图的估计结果文件保存路径
SV_FILE = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results/in_estimateW_result.txt"
# 用户指定每个查询图中推理谓词节点
INFER_NODE_FILE = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph/infer_node.txt"
# 图中节点 ID 与原始CSV文件中数据id的映射关系文件
IDMAP_FILE = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/data_graph/id_mapping.csv"
# 原始 post.csv 文件路径
POST_CSV = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/csv_data/post.csv"
# 存放真实结果的目录
GT_RESULT_DIR = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/ground_truth/structure_result"
# 存放本原型系统结果的目录
OUTPUT_DIR = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results"
# 输出的包含每个查询post估计值w（x）的 post CSV 文件路径
POST_WITH_ESTIMATE_CSV = os.path.join(OUTPUT_DIR, "post_with_estimate.csv")
# 各方法结果的总结文件路径
SUMMARY_CSV = os.path.join(OUTPUT_DIR, "results_summary.csv")
SUMMARY_TXT = os.path.join(OUTPUT_DIR, "results_summary.txt")
# 是否保留临时 CSV（用于调试）
KEEP_TEMP = False  
# ===========================

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------
# Main pipeline
# ---------------------------



dataset_one


In [6]:

print("[BEGIN] multi-query pipeline")

merger = FastestEstimateMerger(
    sv_file=SV_FILE,
    map_file=IDMAP_FILE,
    post_file=POST_CSV,
    output_file=POST_WITH_ESTIMATE_CSV
)
# 1) parse sv multi
sv_df = merger.parse_sv_multi(SV_FILE)
if sv_df is None or sv_df.empty:
    print("[ERROR] No sv records parsed. Exiting.")

# 2) read infer node list
infer_nodes = merger.read_infer_node_list(INFER_NODE_FILE)

# 3) build post_with_estimate.csv
merged_df = merger.build_post_with_estimates(sv_df=sv_df, idmap_file=IDMAP_FILE, post_csv=POST_CSV,
                                     out_csv=POST_WITH_ESTIMATE_CSV)



[BEGIN] multi-query pipeline
[INFO] parse_sv_multi: parsed 222994 node records across 16 queries.
[INFO] read_infer_node_list: loaded 16 infer nodes (first 10): ['u1', 'u2', 'u2', 'u1', 'u2', 'u1', 'u2', 'u2', 'u1', 'u2']
[INFO] build_post_with_estimates: wrote /home/wangshuo/resource/datasets/parler_data/dataset_one/results/post_with_estimate.csv, total rows=24082


### 4. 解析 Fastest 输出结果： 基于单推理谓词也就是root_label各个节点的权重估计值进行proxy指导采样

In [10]:
from pythonProject.src.Structure_first.proxy_sample import compute_T_true_polars
# 定义统计结果保存路径 (确保是全局变量或在函数内定义)
SAMPLED_COUNT_FILE = os.path.join(OUTPUT_DIR, "efficiency/sampled_node_count.csv")

def save_node_counts(records: List[Dict]):
    """辅助函数：将节点计数追加到 CSV"""
    if not records: return
    df = pd.DataFrame(records)
    header = not os.path.exists(SAMPLED_COUNT_FILE)
    try:
        df.to_csv(SAMPLED_COUNT_FILE, mode='a', index=False, header=header)
    except Exception as e:
        print(f"[错误] 写入节点统计失败: {e}")

# 评估参数
RUN_TIMES = 2  # 每种方法重复次数
# ---------------------------
# Step D: 对每个 Query 计算 T_true 并评估
# ---------------------------
print(dataset_name)
def evaluate_queries(
    merged_df: pd.DataFrame,
    sv_df: pd.DataFrame,
    infer_nodes: List[str],
    idmap_file: str,
    post_csv: str,
    gt_result_dir: str,
    output_dir: str,
    run_times: int = 10,
    new_workload: bool = False,  # ✅ 新增：是否重新计算所有 T_true
    proxy_model = 'ML1_proxy4b1_probability',
    T_true_cache_file: str = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}/ground_truth/T_true.txt"  # ✅ 新增：缓存文件路径    
)  -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    对每个 query（按 sv_df 中 query_index 出现顺序）：
      - 使用 infer_nodes 对应的 gt_match_col 计算 T_true（若找不到 GT 文件则 T_true=0）
      - 将对应的 estimate__col 作为 'estimate' 列写入临时 csv
      - 用 ProxyStratifiedSampler 执行四种方法（每种重复 run_times 次），取均值与 std
      - 返回 summary dataframe
    """
    rows = []
    txt_lines = []
    all_node_stats_records = []
    
    # ✅ Step 1: 读取/初始化 T_true 缓存
    if os.path.exists(T_true_cache_file) and not new_workload:
        with open(T_true_cache_file, "r") as f:
            try:
                T_true_cache = json.load(f)
                print(f"[INFO] 已加载缓存的 T_true，共 {len(T_true_cache)} 条。")
            except json.JSONDecodeError:
                print(f"[WARN] {T_true_cache_file} 格式错误，将重新计算所有 T_true。")
                T_true_cache = {}
    else:
        T_true_cache = {}

    # order queries by index
    q_order = sv_df[["query_index", "query_basename"]].drop_duplicates().sort_values("query_index")
    q_order = q_order.reset_index(drop=True)

    n_queries = len(q_order)
    if len(infer_nodes) < n_queries:
        print(f"[WARN] infer_nodes length {len(infer_nodes)} < number of parsed queries {n_queries}. We'll map as many as available and default 'u1' for missing.")
    
    # ✅ Step 2: 遍历每个 query
    for i, r in q_order.iterrows():
        qi = int(r["query_index"])
        qbase = r["query_basename"]
        colname = f"estimate__{qi}__{qbase}"
        print("\n" + "="*60)
        print(f"[STEP] Query index={qi}, basename={qbase}, estimate column={colname}")

        # choose gt_match_col from infer_nodes by index (if available)
        gt_match_col = infer_nodes[i] if i < len(infer_nodes) else "u1"
        print(f"[INFO] Using gt_match_col = {gt_match_col} for query #{qi}")

        # ✅ Step 3: 读取或计算 T_true
        if (not new_workload) and (qbase in T_true_cache):
            T_true = float(T_true_cache[qbase])
            print(f"[CACHE] 读取缓存 T_true={T_true:.4f} for {qbase}")
        else:
            # locate GT file for this query
            # attempt a few plausible filenames in gt_result_dir
            candidates = [
                os.path.join(gt_result_dir, f"{qbase}_matches.csv"),
                os.path.join(gt_result_dir, f"{qbase}.graph_matches.csv"),
                os.path.join(gt_result_dir, f"{qbase}.matches.csv"),
                os.path.join(gt_result_dir, qbase),
            ]
            gt_path = None
            for p in candidates:
                if p and os.path.exists(p):
                    gt_path = p
                    break
            if gt_path is None:
                # try to find any file in gt_result_dir that contains the qbase as substring
                for fname in os.listdir(gt_result_dir):
                    if qbase in fname:
                        cand = os.path.join(gt_result_dir, fname)
                        if os.path.isfile(cand):
                            gt_path = cand
                            break
            if gt_path is None:
                print(f"[WARN] Ground-truth file for query {qbase} not found in {gt_result_dir}. Will use T_true=0.")
                T_true = 0.0
            else:
                try:
                    # compute_T_true should be available (user provided)
                    T_true = compute_T_true_polars(
                        gt_path=gt_path,
                        id_mapping_path=idmap_file,
                        post_csv_path=post_csv,
                        gt_match_col=gt_match_col,
                        prob_col="ML1_oracle1_probability",
                        prob_threshold=0.5
                    )
                except Exception as e:
                    print(f"[ERROR] compute_T_true_polars failed for {qbase} with gt_match_col={gt_match_col}: {e}")
                    traceback.print_exc()
                    T_true = 0.0
            # ✅ 写入缓存
            T_true_cache[qbase] = float(T_true)
            with open(T_true_cache_file, "w") as f:
                json.dump(T_true_cache, f, indent=2)
            print(f"[CACHE] 已更新缓存: {qbase} -> {T_true:.4f}")
            
        # Step 4: 临时 CSV prepare temp CSV for sampler: copy merged_df and set 'estimate' = that col
        if colname not in merged_df.columns:
            print(f"[WARN] estimate column {colname} not found in merged_df. Using zeros.")
            tmp_df = merged_df.copy()
            tmp_df["estimate"] = 0.0
        else:
            tmp_df = merged_df.copy()
            tmp_df["estimate"] = tmp_df[colname].astype(float).fillna(0.0)

        # create temp csv file
        tmp_csv = os.path.join(output_dir, f"tmp_post_with_estimate_q{qi}__{qbase}.csv")
        tmp_df.to_csv(tmp_csv, index=False)
        if not KEEP_TEMP:
            remove_tmp = True
        else:
            remove_tmp = False

        # instantiate sampler (user-provided)
        # sampler = ProxyStratifiedSampler(csv_path=tmp_csv, T_true=T_true,proxy_model=proxy_model)
        sampler = ProxyStratifiedSampler(csv_path=tmp_csv, T_true=T_true,is_multi_predicate=False, # 明确告知这是单谓词场景
                                         post_proxy=proxy_model,     # 将 proxy_model 赋值给 post_proxy
                                        total_budget_frac=0.1
                                        )

        methods = {
            # ✅ 新增三种 baseline
            "baseline_uniform": sampler.run_baseline_uniform,
            # "baseline_proxy": sampler.run_baseline_proxy,
            "baseline_proxy_a": sampler.run_baseline_proxy_a,
            "baseline_graph_only": sampler.run_baseline_graph_only,
            # "baseline_a": sampler.run_baseline_a,                   # proportional to a
            # "baseline_proxy_mul_a": sampler.run_baseline_proxy_mul_a, # proportional to proxy * a
            # 原有四种分层采样法
            "proxy_importance": sampler.run_proxy_importance,
            # "proxy_uniform": sampler.run_proxy_uniform,
            "proxyE_importance": sampler.run_proxyE_importance,
            # "proxy_importance_pilot_IS": sampler.run_proxy_importance_pilot_is,
            # "proxyE_uniform": sampler.run_proxyE_uniform
        }

        for mname, func in methods.items():
            T_list = []
            Q_list = []
            post_cnt_list = []
            comment_cnt_list = [] # 单谓词下通常为0
            # +++ 新增：收集 pi 统计信息 +++
            pi_mean_list = []
            pi_min_list = []
            pi_max_list = []
            
            print(f"\n--- Running {mname} for {run_times} times ---")
            for t in range(run_times):
                try:
                    out = func()
                    T_hat = float(out.get("T_hat", 0.0))
                    Qerror = float(out.get("Qerror", 1.0))
                    T_list.append(T_hat)
                    Q_list.append(Qerror)
                    # ✅ 实时打印每次结果
                    # print(f"[{mname} | run {t+1}/{run_times}]  T_hat={T_hat:.4f},  Qerror={Qerror:.6f}")
                    post_cnt_list.append(out.get("n_post", 0))
                    comment_cnt_list.append(out.get("n_comment", 0))
                    pi_mean_list.append(out.get("pi_mean", 0.0))
                    pi_min_list.append(out.get("pi_min", 0.0))
                    pi_max_list.append(out.get("pi_max", 0.0))
                except Exception as e:
                    print(f"❌ {mname} 第 {t+1} 次执行失败: {repr(e)}")
                    traceback.print_exc()
                    T_list.append(0.0)
                    Q_list.append(1.0)
                    list.append(0)
                    comment_cnt_list.append(0)

            # # 计算均值与标准差
            # T_mean = float(np.mean(T_list))
            # T_std = float(np.std(T_list))
            # Q_mean = float(np.mean(Q_list))
            # Q_std = float(np.std(Q_list))
            # 去掉最高值和最低值计算均值和标准差
            def trimmed_mean_std(lst):
                if len(lst) <= 2:
                    return np.mean(lst), np.std(lst)
                sorted_lst = sorted(lst)
                trimmed = sorted_lst[1:-1]
                return np.mean(trimmed), np.std(trimmed)
            
            T_mean, T_std = trimmed_mean_std(T_list)
            Q_mean, Q_std = trimmed_mean_std(Q_list)
            avg_post = int(np.mean(post_cnt_list))
            avg_comment = int(np.mean(comment_cnt_list))
            
            avg_pi_mean = np.mean(pi_mean_list)
            avg_pi_min = np.mean(pi_min_list)
            avg_pi_max = np.mean(pi_max_list)

            print(f"✅ {mname} 平均结果: T_hat={T_mean:.4f}±{T_std:.4f},  Qerror={Q_mean:.6f}±{Q_std:.6f}")
            print(f"   Samples(P/C)={avg_post}/{avg_comment} | Pi(Mean/Min/Max)={avg_pi_mean:.4f} / {avg_pi_min:.4f} / {avg_pi_max:.4f}")
            # 写入结果表
            rows.append({
                "query_index": qi,
                "query_basename": qbase,
                "gt_match_col": gt_match_col,
                "T_true": float(T_true),
                "method": mname,
                "T_hat_mean": T_mean,
                "T_hat_std": T_std,
                "Qerror_mean": Q_mean,
                "Qerror_std": Q_std
            })

            # 同时写入 TXT 文本（summary）
            txt_lines.append(
                f"{qbase} {gt_match_col} {mname} "
                f"T_hat={T_mean:.6f}±{T_std:.6f} "
                f"Qerror={Q_mean:.6f}±{Q_std:.6f}"
            )
            # 记录节点采样统计
            all_node_stats_records.append({
                "query_name": qbase,
                "method": mname,
                "post_sampled_cnt": avg_post,
                "comment_sampled_cnt": avg_comment
            })

        # cleanup tmp csv
        if remove_tmp:
            try:
                os.remove(tmp_csv)
            except Exception:
                pass

    # write summary files
    df_summary = pd.DataFrame(rows)
    df_summary.to_csv(SUMMARY_CSV, index=False)
    with open(SUMMARY_TXT, "w") as f:
        for ln in txt_lines:
            f.write(ln + "\n")
    if all_node_stats_records:
        save_node_counts(all_node_stats_records)
        print(f"[INFO] 已保存节点采样统计到 {SAMPLED_COUNT_FILE}")
    print(f"[INFO] evaluate_queries: wrote summary to {SUMMARY_CSV} and {SUMMARY_TXT}")
    return df_summary,all_node_stats_records


# 4) evaluate each query
summary_df, df_node_stats = evaluate_queries(
    merged_df=merged_df,
    sv_df=sv_df,
    infer_nodes=infer_nodes,
    idmap_file=IDMAP_FILE,
    post_csv=POST_CSV,
    gt_result_dir=GT_RESULT_DIR,
    output_dir=OUTPUT_DIR,
    run_times=RUN_TIMES,
    proxy_model='ML1_proxy4b1_probability',
)

print("[END] pipeline completed")
if summary_df is not None:
    print(summary_df.head())


dataset_one
[INFO] 已加载缓存的 T_true，共 23 条。

[STEP] Query index=0, basename=query_dense_1_1.graph, estimate column=estimate__0__query_dense_1_1.graph
[INFO] Using gt_match_col = u1 for query #0
[CACHE] 读取缓存 T_true=16656.0000 for query_dense_1_1.graph
[Check_total_budget_frac] 0.1

--- Running baseline_uniform for 2 times ---
✅ baseline_uniform 平均结果: T_hat=16223.4576±969.5299,  Qerror=0.058209±0.025969
   Samples(P/C)=979/0 | Pi(Mean/Min/Max)=0.0999 / 0.0999 / 0.0999

--- Running baseline_proxy_a for 2 times ---
[Proxy×A] weights: min=0.011019, max=19.950151, mean=0.544165
[Proxy×A] probs  : min=0.000002,  max=0.003742,  mean=0.000102
[Proxy×A] weights: min=0.011019, max=19.950151, mean=0.544165
[Proxy×A] probs  : min=0.000002,  max=0.003742,  mean=0.000102
✅ baseline_proxy_a 平均结果: T_hat=15669.5197±235.9157,  Qerror=0.059227±0.014164
   Samples(P/C)=979/0 | Pi(Mean/Min/Max)=0.4384 / 0.0022 / 1.0000

--- Running baseline_graph_only for 2 times ---
✅ baseline_graph_only 平均结果: T_hat=16223.122